In [8]:
# We have to write funtions manually to calculate for all equation
def dy_dx(x):
    return 2 * x
print("Derivative of x^2 ->",dy_dx(3))

Derivative of x^2 -> 6


In [ ]:
# How to use Autograd
import torch
x = torch.tensor(3.0, requires_grad=True)
y = x**2
print(f"Orginal x: {x}, Original y: {y}")

# y is root node
y.backward()

# x is leaf node we can only see the grad attribute of root node
# otherwise we will see this error -> The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead.
print(f"Gradient of y: {x.grad}")

Orginal x: 3.0, Original y: 9.0
Gradient of y: 6.0


In [ ]:
# Lets see How the tensors look with requires_grad=True
# grad can be implicitly created only for scalar outputs Not for vectors
x = torch.tensor(
    [1, 2, 3], dtype=torch.float32, requires_grad=True
)                               # tensor([1., 2., 3.], requires_grad=True)

y = x**2                        # tensor([1., 4., 9.], grad_fn=<PowBackward0>)
z = torch.sin(y).mean()         # tensor([ 0.8415, -0.7568,  0.4121], grad_fn=<SinBackward0>)

z.backward()
print(f"Gradient of z w.r.t x: {x.grad}")


Gradient of z w.r.t x: tensor([ 0.3602, -0.8715, -1.8223])


In [30]:
# Example of Forward Propogation and then calculating the derivative of LOSS
# Binary Cross Entropy

x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias


# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(
        target * torch.log(prediction)
        + (1 - target) * torch.log(1 - prediction)
    )

# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [32]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y) / (y_pred * (1 - y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


In [55]:
# Now calculate with the help of autograd
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)


In [61]:
# If we run this shell multiple times we will see that gradient will start accumulation
# they will not reset by default for that we have to perform clear_gard operation
# which can be done by 3 ways 
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()
z = w * x + b
y_pred = torch.sigmoid(z)
loss = binary_cross_entropy_loss(y_pred, y)
loss.backward()

print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


In [62]:
w.grad.zero_() # setting grad to zero So it will not accumulate
x.requires_grad_(False) # Now x will look like  this: tensor(6.7000)

tensor(6.7000)

In [63]:
# detach tensor from original tensor
x.detach()  # Now x will look like this: tensor(6.7000)

tensor(6.7000)